In [24]:
import os

from physics.simulation import msq, sample
from physics.hzz import zpair, zz4l
from alice import dataset, model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.colors import LogNorm
import hist

import torch
from torch.utils.data import TensorDataset
from lightning import Trainer

In [25]:
OUTPUT_DIR = 'jobs/alice/run-one-copy'
SCALER_FILE = 'scaler.pkl'
CHECKPOINT_DIR = 'checkpoints'
SAMPLE_DIR = '..'

CHECKPOINT = 'checkpoint-alice-epoch=27-val_loss=0.007.ckpt'

VERSION = 2
LIGHTNING_DIR = f'lightning_logs/version_{VERSION}'

COMPONENT_1 = msq.Component.SIG
COMPONENT_2 = msq.Component.BKG

SAMPLE_SIZE = 100000
BATCH_SIZE = 64
SEED = 42

In [26]:
xs = {
    msq.Component.SBI : 1.5569109,
    msq.Component.SIG : 0.15105108,
    msq.Component.INT : -0.22043824,
    msq.Component.BKG : 1.6270497
}

filenames = {
    msq.Component.SBI : 'ggZZ2e2m_sbi.csv',
    msq.Component.SIG : 'ggZZ2e2m_sig.csv',
    msq.Component.INT : 'ggZZ2e2m_int.csv',
    msq.Component.BKG : 'ggZZ2e2m_bkg.csv'
}

In [27]:
sample_background = sample.from_csv(cross_section=xs[COMPONENT_2], file_path=os.path.join(SAMPLE_DIR, filenames[COMPONENT_1]), n_rows=int(SAMPLE_SIZE*1.2))
        
msq_bkg_null = msq.MSQFilter('msq_bkg_sm', 0.0)
msq_bkg_nan = msq.MSQFilter('msq_bkg_sm', np.nan)

z_cand = zpair.ZPairCandidate(algorithm='leastsquare')
z_masses = zpair.ZPairMassWindow(z1=(70,115), z2=(70,115))
angles = zz4l.AngularVariables()
four_lepton_vars = zz4l.FourLeptonSystem()

sample_processed = sample_background.filter(msq_bkg_nan).filter(msq_bkg_null).calculate(z_cand).filter(z_masses).calculate(angles).calculate(four_lepton_vars)[:SAMPLE_SIZE]

features = ['cth_star', 'cth_1', 'cth_2', 'phi_1', 'phi', 'Z1_mass', 'Z2_mass', '4l_mass', '4l_rapidity']

# Get only required features
X = sample_processed.kinematics[features].to_numpy()

# Get PDF ratios for p(theta_0)/p(theta_1)
pdf_ratios = sample_processed.probabilities/sample_processed.reweight(numerator=COMPONENT_1, denominator=COMPONENT_2).probabilities

y = (1/(1 + pdf_ratios)).to_numpy()

# Use N*(p(theta_1) + p(theta_0))/2 as sample weights
sample_weights = SAMPLE_SIZE*(sample_processed.probabilities + sample_processed.reweight(numerator=COMPONENT_1, denominator=COMPONENT_2).probabilities).to_numpy()/2

data = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), torch.tensor(sample_weights, dtype=torch.float32))

In [28]:
model_alice = model.ALICE.load_from_checkpoint(os.path.join(OUTPUT_DIR, CHECKPOINT_DIR, CHECKPOINT))

In [29]:
predictions = model_alice(data.tensors[0]).detach().view(-1).numpy()
ratio_predicted = predictions/(1-predictions)

In [30]:
SIG_weights_predicted = sample_processed.weights * ratio_predicted
SIG_weights_reweight = sample_processed[COMPONENT_1].weights

TypeError: Cannot index by location index with a non-integer key